In [1]:
import sys
sys.path.append("/home/landelle/pytorch-cifar/models")
PATH_TO_EFFICIENT_NET = "/home/landelle/EfficientNet-PyTorch"
sys.path.append(PATH_TO_EFFICIENT_NET)

import random
import numpy as np
import pandas as pd
import torch
import torchvision
import datetime

# Local files
import resnet
import main

In [2]:

SEED = 2019
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark = False

In [3]:

transform = torchvision.transforms.transforms.Compose([
    torchvision.transforms.transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(root='datasets', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='datasets', train=False, download=True, transform=transform)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
trainset, testset

Files already downloaded and verified
Files already downloaded and verified


(Dataset CIFAR10
     Number of datapoints: 50000
     Root location: datasets
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
            ), Dataset CIFAR10
     Number of datapoints: 10000
     Root location: datasets
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
            ))

In [4]:

STATE_DICTS_PATH = "/home/landelle/state_dicts/"
STATE_DICTS_PREFIX = "state_dict_resnet18"

BATCH_SIZE = 100
N_BATCHES_IN_TRAIN_SET = len(trainset) // BATCH_SIZE
N_BATCHES_IN_TEST_SET = len(testset) // BATCH_SIZE

NUM_WORKERS = 8

# Fixed learning rate
LR = 0.1 #0.005 LR=0.1 is used by kuangliu with his implementation of ResNet18 for epochs [0;150)

# CIFAR10 number of classes
NUM_CLASS = 10

# CIFAR10 image metadata
CHANNEL, IMAGE_SIZE, _ = trainset[0][0].shape
print("images are:", IMAGE_SIZE, CHANNEL)

TRAIN_EPOCHS = 5
print(N_BATCHES_IN_TRAIN_SET)

images are: 32 3
500


In [5]:

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [6]:

def get_model():
    """ Get model is used to remake the model between lambda attempts """
    return resnet.ResNet18()

def get_print(save_raw=False, f_path="results"):
    """ Makes print function to allow keeping output on disk """
    def print_(*a, **kwa):
        if save_raw:
            with open(f_path, "a") as f:
                t = datetime.datetime.now()
                f.write(str(t) + '    ' + ' '.join([str(x) for x in a]) + '\n')
        print(*a, **kwa)
    return print_

In [7]:

def train(device, model, optimizer, criterion, lam, n_epochs=None, n_batches=None,
          save_raw=False, save_state_dicts="", save_tensorboard=""):
    """
    save_raw: save raw results (also print to external file) 
    save_state_dicts: No|Override|Separate, saves state_dicts after an epoch
    save_tensorboard: saves tensorboard data
    """
    print_ = get_print(save_raw)
    
    # switch to train mode
    model.train()
    n_epochs = n_epochs if n_epochs else TRAIN_EPOCHS
    for epoch in range(1, n_epochs + 1):
        print_("Epoch[{}/{}]".format(epoch, n_epochs))
        for batch_id, (images, labels) in enumerate(trainloader):
            if n_batches and batch_id > n_batches:
                break            
            labels, images = labels.to(device), images.to(device)
            images, labels_a, labels_b, lam = main.mixup_data(images, labels, lam)
            outputs = model(images)
            loss = main.mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_id % 50 == 0:
                print_('Loss :{:.4f} Epoch[{}/{}] Batch[{}/{}] batch_shape:{}'.format(
                    loss.item(), epoch, n_epochs, batch_id, N_BATCHES_IN_TRAIN_SET, images.shape))
        # Each epoch if enabled: save state dicts
        if save_state_dicts=="Separate":
            torch.save(model.state_dict(), STATE_DICTS_DIR + STATE_DICTS_PREFIX + "_epoch_" + str(epoch))
        elif save_state_dicts=="Override":
            torch.save(model.state_dict(), STATE_DICTS_DIR + STATE_DICTS_PREFIX + "_override")
            

In [8]:

def test(device, model, save_raw=False):
    print_ = get_print(save_raw)
        
    # switch to evaluate mode
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in testloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        # Output test accuracy
        test_acc = 100 * correct / total
        print_('Test Accuracy of the model on the test images: {} %'.format(test_acc))
        return test_acc

In [12]:

def collect_results(n_epochs=10, n_batches=None, half=False, no_mixup=False,
                    save_raw=False, save_state_dicts=False, save_tensorboard=False, custom_lambdas=[]):
    """
    n_batches: None trains on whole dataset otherwise it trains on n_batches of BATCH_SIZE per epoch    
    
    """
    USE_CUDA = True
    print_ = get_print(save_raw)

    test_accs = {}
    for lambda_ in custom_lambdas if custom_lambdas else [x*.1 for x in range(11)]:
        if half and lambda_>.5:
            break
        if no_mixup and lambda_>0:
            break
        print_("Trying lambda=", lambda_)        

        device = torch.device('cuda' if USE_CUDA else 'cpu')
        #model = models.My_Model(NUM_CLASS).to(device)
        #model = EfficientNet.from_pretrained('efficientnet-b0').to(device)
        model = get_model().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = torch.nn.CrossEntropyLoss()

        train(device, model, optimizer, criterion, lambda_, n_epochs, n_batches, save_raw, save_state_dicts, save_tensorboard)
        test_accs[lambda_] = test(device, model, save_raw)

        del device, model, optimizer, criterion
    test_accs_dict = {str(k)[:3]:[v] for k, v in test_accs.items()}
    test_accs_df = pd.DataFrame.from_dict(test_accs_dict).transpose()
    test_accs_df.reset_index(inplace=True)
    test_accs_df.rename(columns={0:'Test accuracy', 'index':'Lambda'}, inplace=True)
    print_(test_accs_df)
    return test_accs_df



In [22]:
CUSTOM_LAMBDAS = [x*.25 for x in range(7)]
print(CUSTOM_LAMBDAS)

SAVE_RAW = True
if SAVE_RAW:
    with open("results", "w") as f: pass
test_accs2 = collect_results(n_epochs=100, n_batches=None, save_raw=SAVE_RAW, custom_lambdas=CUSTOM_LAMBDAS)

[0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
Trying lambda= 0.0
Epoch[1/100]
Loss :2.4788 Epoch[1/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :2.3131 Epoch[1/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :2.2967 Epoch[1/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :2.1316 Epoch[1/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :2.1275 Epoch[1/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.9639 Epoch[1/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.9385 Epoch[1/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.7386 Epoch[1/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.8164 Epoch[1/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.7086 Epoch[1/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[2/100]
Loss :1.8647 Epoch[2/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :1.7058 Epoch[

Loss :0.4844 Epoch[10/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.6050 Epoch[10/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[11/100]
Loss :0.3818 Epoch[11/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4311 Epoch[11/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.5166 Epoch[11/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.3356 Epoch[11/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.5175 Epoch[11/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4227 Epoch[11/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4481 Epoch[11/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4680 Epoch[11/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4257 Epoch[11/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.4255 Epoch[11/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32

Loss :0.2538 Epoch[20/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1164 Epoch[20/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1839 Epoch[20/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.2328 Epoch[20/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[21/100]
Loss :0.0968 Epoch[21/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.2255 Epoch[21/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0756 Epoch[21/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0833 Epoch[21/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.2116 Epoch[21/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1214 Epoch[21/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1417 Epoch[21/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0929 Epoch[21/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32

Loss :0.0395 Epoch[30/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0172 Epoch[30/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.2102 Epoch[30/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1394 Epoch[30/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0208 Epoch[30/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0752 Epoch[30/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[31/100]
Loss :0.0507 Epoch[31/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1306 Epoch[31/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0431 Epoch[31/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0250 Epoch[31/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.1195 Epoch[31/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0313 Epoch[31/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32

Loss :0.0394 Epoch[40/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0132 Epoch[40/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0537 Epoch[40/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0289 Epoch[40/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0464 Epoch[40/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0399 Epoch[40/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0101 Epoch[40/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0299 Epoch[40/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[41/100]
Loss :0.0109 Epoch[41/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0486 Epoch[41/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0135 Epoch[41/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :0.0170 Epoch[41/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32

...

Loss :-238382272.0000 Epoch[62/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-193307312.0000 Epoch[62/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-233897344.0000 Epoch[62/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-224912496.0000 Epoch[62/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-172548752.0000 Epoch[62/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[63/100]
Loss :-255464800.0000 Epoch[63/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-224413200.0000 Epoch[63/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-212444384.0000 Epoch[63/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-230149040.0000 Epoch[63/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-198873280.0000 Epoch[63/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-177876688.0000 Epoch[63/100] Batch[250/500] batch_shape:torch.

Loss :-279400704.0000 Epoch[71/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-287236768.0000 Epoch[71/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-335788320.0000 Epoch[71/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-355865632.0000 Epoch[71/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-295380000.0000 Epoch[71/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-339067936.0000 Epoch[71/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-320990560.0000 Epoch[71/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[72/100]
Loss :-307525376.0000 Epoch[72/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-303138176.0000 Epoch[72/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-271207872.0000 Epoch[72/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-348075552.0000 Epoch[72/100] Batch[150/500] batch_shape:torch.

Loss :-374882336.0000 Epoch[80/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-414783616.0000 Epoch[80/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-411011008.0000 Epoch[80/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-508068640.0000 Epoch[80/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-379647584.0000 Epoch[80/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-383563744.0000 Epoch[80/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-453788256.0000 Epoch[80/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-472651808.0000 Epoch[80/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-379079168.0000 Epoch[80/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[81/100]
Loss :-401621376.0000 Epoch[81/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-446151648.0000 Epoch[81/100] Batch[50/500] batch_shape:torch.S

Epoch[89/100]
Loss :-539785152.0000 Epoch[89/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-497307008.0000 Epoch[89/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-599705920.0000 Epoch[89/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-470043264.0000 Epoch[89/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-515174784.0000 Epoch[89/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-578939904.0000 Epoch[89/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-510881120.0000 Epoch[89/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-646035904.0000 Epoch[89/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-547020224.0000 Epoch[89/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-574650624.0000 Epoch[89/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[90/100]
Loss :-507346848.0000 Epoch[90/100] Batch[0/500] batch_

Loss :-707665280.0000 Epoch[97/100] Batch[400/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-754152704.0000 Epoch[97/100] Batch[450/500] batch_shape:torch.Size([100, 3, 32, 32])
Epoch[98/100]
Loss :-637379328.0000 Epoch[98/100] Batch[0/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-643433920.0000 Epoch[98/100] Batch[50/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-730935296.0000 Epoch[98/100] Batch[100/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-612378688.0000 Epoch[98/100] Batch[150/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-650950144.0000 Epoch[98/100] Batch[200/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-709972288.0000 Epoch[98/100] Batch[250/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-794822464.0000 Epoch[98/100] Batch[300/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-823670848.0000 Epoch[98/100] Batch[350/500] batch_shape:torch.Size([100, 3, 32, 32])
Loss :-733808960.0000 Epoch[98/100] Batch[400/500] batch_shape:torch.

In [ ]:
test_accs2